In [8]:
import time
import numpy as np

# класс детектор из первой задачи
class Detector:
    def __init__(self, R=2.0, N=40, r=0.1, a=0.5):
        self.R = R # радиус детектора в метрах[cite: 1]
        self.N = N # количество ФЭУ[cite: 1]
        self.r = r # радиус фотокатода ФЭУ в метрах[cite: 1]
        self.a = a # параметр пуассона[cite: 1, 2]

        self.pmt_positions = self.generate_pmt_positions()
        self.pmt_angular_radius = np.arcsin(self.r / self.R)

    # равномерное распределение точек на сфере
    def generate_pmt_positions(self):
        positions = []
        phi = np.pi * (np.sqrt(5.0) - 1.0)  

        for i in range(self.N):
            y = 1.0 - (i / float(self.N - 1)) * 2.0  # y от 1 до -1
            radius_at_y = np.sqrt(1.0 - y * y)
            theta = phi * i

            x = np.cos(theta) * radius_at_y
            z = np.sin(theta) * radius_at_y

            pos = np.array([x, y, z]) * self.R
            positions.append(pos)
            
        return np.array(positions)

    # метод для определения принадлежности точки
    def check_hits_vectorized(self, hit_points):
        norm_hits = hit_points / np.linalg.norm(hit_points, axis=1, keepdims=True)
        norm_pmts = self.pmt_positions / self.R  
        
        cos_angles = np.dot(norm_hits, norm_pmts.T)  
        cos_angles = np.clip(cos_angles, -1.0, 1.0)
        angles = np.arccos(cos_angles)
        
        hits_mask = angles <= self.pmt_angular_radius
        hit_indices = np.argmax(hits_mask, axis=1)
        has_hit = np.any(hits_mask, axis=1)
        hit_indices[~has_hit] = -1
        
        valid_hits = hit_indices[hit_indices != -1]
        pmt_hits = np.bincount(valid_hits, minlength=self.N)
        return pmt_hits


# класс источник из второй задачи
class Source:
    def __init__(self, Y=10000):
        self.Y = Y  # световыход: фотонов / МэВ

    def generate_photons(self, energy, pos=(0.0, 0.0, 0.0)):
        # общее число фотонов пропорционально энергии и световыходу Y
        n_photons = int(energy * self.Y)
        
        # изотропное распределение направлений 
        phi = np.random.uniform(0, 2 * np.pi, n_photons)
        costheta = np.random.uniform(-1.0, 1.0, n_photons)
        sintheta = np.sqrt(1.0 - costheta**2)

        nx = sintheta * np.cos(phi)
        ny = sintheta * np.sin(phi)
        nz = costheta

        directions = np.column_stack((nx, ny, nz))
        positions = np.tile(pos, (n_photons, 1))
        
        return positions, directions


# Моделирование распространения и детектирования света 
def simulate_single_event(detector, source, energy, source_pos=(0.0, 0.0, 0.0)):
    r0, directions = source.generate_photons(energy, pos=source_pos)

    # расчет точек пересечения лучей со сферой радиуса R
    b = 2 * np.sum(r0 * directions, axis=1)
    c = np.sum(r0**2, axis=1) - detector.R**2
    t = (-b + np.sqrt(b**2 - 4 * c)) / 2.0
    
    hit_points = r0 + directions * t[:, np.newaxis]
    
    # считаем попадания по каналам ФЭУ
    pmt_hits = detector.check_hits_vectorized(hit_points)
    
    # отклик фотокатода (распределение пуассона со средним a * hits)
    pe_per_pmt = np.random.poisson(detector.a * pmt_hits)
    total_charge = np.sum(pe_per_pmt)
    
    return total_charge  


# энергетический отклик
detector = Detector()
source = Source()

energies = [1.0, 0.2, 0.5, 0.7, 1.5, 2.0, 3.0]
n_events = 1000  

mean_1v = 0.0 # переменная для сохранения результата 1 МэВ

for E in energies:
    charges = [simulate_single_event(detector, source, E, source_pos=(0, 0, 0)) for _ in range(n_events)]
    mean_q = np.mean(charges)
    std_q = np.std(charges)
    resolution = std_q / mean_q
    
    # сохраняем значение для 1.0 МэВ для теста смещения
    if E == 1.0:
        mean_1v = mean_q
    
    print(f"Энергия: {E:3.1f} МэВ | Средний заряд: {mean_q:6.1f} ф.э. | СКО: {std_q:5.1f} | Разрешение: {resolution:.4f}")


# смещение источника на 1 метр по оси z
print("\nТест смещения источника на 1 м по оси z (E = 1.0 МэВ)")
shifted_charges = [simulate_single_event(detector, source, 1.0, source_pos=(0.0, 0.0, 1.0)) for _ in range(n_events)]
mean_shifted = np.mean(shifted_charges)

print(f"Источник в центре (0,0,0): средний заряд = {mean_1v:.1f} ф.э.")
print(f"Источник смещен (0,0,1): средний заряд = {mean_shifted:.1f} ф.э.")
print(f"Изменение среднего значения: {abs(mean_shifted - mean_1v) / mean_1v * 100:.2f}%")

Энергия: 1.0 МэВ | Средний заряд:  125.0 ф.э. | СКО:  13.9 | Разрешение: 0.1108
Энергия: 0.2 МэВ | Средний заряд:   25.0 ф.э. | СКО:   5.9 | Разрешение: 0.2355
Энергия: 0.5 МэВ | Средний заряд:   62.3 ф.э. | СКО:   9.4 | Разрешение: 0.1509
Энергия: 0.7 МэВ | Средний заряд:   87.0 ф.э. | СКО:  11.2 | Разрешение: 0.1287
Энергия: 1.5 МэВ | Средний заряд:  186.6 ф.э. | СКО:  16.7 | Разрешение: 0.0893
Энергия: 2.0 МэВ | Средний заряд:  249.2 ф.э. | СКО:  19.4 | Разрешение: 0.0778
Энергия: 3.0 МэВ | Средний заряд:  376.3 ф.э. | СКО:  23.7 | Разрешение: 0.0631

Тест смещения источника на 1 м по оси z (E = 1.0 МэВ)
Источник в центре (0,0,0): средний заряд = 125.0 ф.э.
Источник смещен (0,0,1): средний заряд = 124.4 ф.э.
Изменение среднего значения: 0.50%
